<a href="https://colab.research.google.com/github/namrathajujjavarapu24-maker/AI-Powered-Visual-Defect-Detection/blob/main/CodeAlpha_FAQChatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio nltk scikit-learn

In [2]:
# ============================================================
# CODEALPHA - AI FAQ ASSISTANT
# Student Support FAQ Chatbot
# NLP + TF-IDF + Cosine Similarity + Gradio
# ============================================================

import gradio as gr
import nltk
import re

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# NLTK SETUP
# ============================================================

nltk.download("stopwords", quiet=True)

STOP_WORDS = set(stopwords.words("english"))


# ============================================================
# FAQ DATABASE
# ============================================================

faqs = [

    {
        "question": "What are the college timings?",
        "answer": "The college is open from 9:00 AM to 4:30 PM, Monday to Friday."
    },

    {
        "question": "What are the library timings?",
        "answer": "The library is open from 8:30 AM to 6:00 PM on working days."
    },

    {
        "question": "How can I apply for a course?",
        "answer": "You can apply for a course through the college admission portal or contact the admissions office."
    },

    {
        "question": "How can I contact the administration?",
        "answer": "You can contact the college administration through the official administration office or college contact details."
    },

    {
        "question": "How can I get my student ID card?",
        "answer": "Students can collect their ID card from the administration or student services office."
    },

    {
        "question": "Where can I find the examination schedule?",
        "answer": "The examination schedule is normally published through the college notice board or official student portal."
    },

    {
        "question": "How can I check my attendance?",
        "answer": "You can check your attendance through the student portal or contact your department office."
    },

    {
        "question": "How can I contact my department?",
        "answer": "You can contact your department office or speak with the department coordinator."
    },

    {
        "question": "How do I apply for leave?",
        "answer": "Students can apply for leave according to the college leave procedure through the appropriate department or student portal."
    },

    {
        "question": "Where can I find the academic calendar?",
        "answer": "The academic calendar can be found on the official college website or student portal."
    },

    {
        "question": "How can I pay my college fees?",
        "answer": "College fees can be paid through the official fee payment portal or through the college accounts office."
    },

    {
        "question": "What should I do if I lose my ID card?",
        "answer": "Report the lost ID card to the administration or student services office and follow their replacement procedure."
    }

]


# ============================================================
# TEXT PREPROCESSING
# ============================================================

def preprocess_text(text):

    text = text.lower()

    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        "",
        text
    )

    words = text.split()

    words = [
        word
        for word in words
        if word not in STOP_WORDS
    ]

    return " ".join(words)


# ============================================================
# PREPARE FAQ QUESTIONS
# ============================================================

faq_questions = [
    faq["question"]
    for faq in faqs
]

processed_questions = [
    preprocess_text(question)
    for question in faq_questions
]


# ============================================================
# TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer()

faq_vectors = vectorizer.fit_transform(
    processed_questions
)


# ============================================================
# FIND BEST FAQ MATCH
# ============================================================

def get_answer(user_question):

    if not user_question or not user_question.strip():

        return (
            "⚠️ Please enter a question first.",
            0,
            "No question entered"
        )

    processed_query = preprocess_text(
        user_question
    )

    query_vector = vectorizer.transform(
        [processed_query]
    )

    similarity_scores = cosine_similarity(
        query_vector,
        faq_vectors
    )[0]

    best_index = similarity_scores.argmax()

    best_score = similarity_scores[
        best_index
    ]

    matched_question = faq_questions[
        best_index
    ]

    matched_answer = faqs[
        best_index
    ]["answer"]

    if best_score < 0.15:

        return (
            "🤔 I couldn't find a suitable answer in my FAQ database. "
            "Please try asking your question in a different way.",
            best_score,
            "No strong FAQ match found"
        )

    return (
        matched_answer,
        best_score,
        matched_question
    )


# ============================================================
# CHAT FUNCTION
# ============================================================

def respond(message, history):

    if history is None:
        history = []

    history = list(history)

    answer, score, matched_question = get_answer(
        message
    )

    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": answer
    })

    confidence = score * 100

    match_info = f"""
### 🎯 AI Matching Result

**Matched FAQ**

{matched_question}

**Cosine Similarity Score**

`{confidence:.1f}%`

**Method**

TF-IDF + Cosine Similarity
"""

    return (
        history,
        "",
        match_info
    )


# ============================================================
# CLEAR CHAT
# ============================================================

def clear_chat():

    return (
        [],
        "",
        "💬 Ask a question to get started."
    )


# ============================================================
# 🍋🌿 PREMIUM CUSTOM CSS
# 70% LEMON YELLOW + 30% PARROT GREEN
# NO RED
# ============================================================

custom_css = """

/* ============================================================
   ROOT COLORS
   ============================================================ */

:root {

    --lemon: #fff200;
    --lemon-bright: #ffff4d;
    --lemon-soft: #fff9a6;
    --lemon-pale: #fffde2;

    --parrot: #62c93d;
    --parrot-bright: #7bea4d;
    --parrot-light: #bdf58a;
    --parrot-pale: #e9fbd5;

    --dark-green: #183800;
    --deep-green: #244900;
    --text-green: #365500;

    --border-yellow: #d9d400;
    --border-green: #83c957;

    --white: #ffffff;
}


/* ============================================================
   BODY
   ============================================================ */

body {

    margin: 0 !important;

    background:

        radial-gradient(
            circle at 8% 8%,
            rgba(255, 242, 0, 0.50),
            transparent 25%
        ),

        radial-gradient(
            circle at 92% 15%,
            rgba(98, 201, 61, 0.30),
            transparent 24%
        ),

        radial-gradient(
            circle at 50% 95%,
            rgba(255, 242, 0, 0.35),
            transparent 30%
        ),

        #fffde7 !important;
}


/* ============================================================
   MAIN GRADIO CONTAINER
   ============================================================ */

.gradio-container {

    max-width: 1180px !important;

    margin: auto !important;

    min-height: 100vh !important;

    padding-bottom: 20px !important;

    font-family:
        Inter,
        "Segoe UI",
        Arial,
        sans-serif !important;

    background:

        radial-gradient(
            circle at 15% 5%,
            rgba(255, 242, 0, 0.32),
            transparent 25%
        ),

        radial-gradient(
            circle at 90% 35%,
            rgba(98, 201, 61, 0.18),
            transparent 25%
        ),

        linear-gradient(
            180deg,
            #fffef0 0%,
            #fffbd1 45%,
            #f5fbdc 100%
        ) !important;
}


/* ============================================================
   HERO
   ============================================================ */

.hero {

    position: relative;

    text-align: center;

    padding:
        65px
        20px
        38px;

    overflow: hidden;
}


/* Hero decorative glow */

.hero::before {

    content: "";

    position: absolute;

    width: 230px;

    height: 230px;

    border-radius: 50%;

    background:
        rgba(255, 242, 0, 0.30);

    filter: blur(55px);

    top: 10px;

    left: 5%;

    z-index: 0;
}


.hero::after {

    content: "";

    position: absolute;

    width: 220px;

    height: 220px;

    border-radius: 50%;

    background:
        rgba(98, 201, 61, 0.22);

    filter: blur(55px);

    right: 5%;

    bottom: 5px;

    z-index: 0;
}


.hero > * {

    position: relative;

    z-index: 2;
}


/* ============================================================
   HERO BADGE
   ============================================================ */

.badge {

    display: inline-flex;

    align-items: center;

    justify-content: center;

    padding:
        10px
        20px;

    border-radius: 50px;

    background:

        linear-gradient(
            135deg,
            #fff200,
            #eaff72
        ) !important;

    border:
        2px solid
        #c9c500 !important;

    color:
        #294400 !important;

    font-size: 12px;

    font-weight: 900;

    letter-spacing: 1.6px;

    box-shadow:

        0 8px 25px
        rgba(190, 190, 0, 0.22);

    transition:
        transform 0.3s ease,
        box-shadow 0.3s ease;
}


.badge:hover {

    transform:
        translateY(-3px);

    box-shadow:

        0 12px 30px
        rgba(130, 170, 40, 0.28);
}


/* ============================================================
   HERO TITLE
   ============================================================ */

.hero h1 {

    margin:
        24px
        0
        14px;

    font-size: 56px;

    font-weight: 950;

    letter-spacing: -1.5px;

    line-height: 1.08;

    background:

        linear-gradient(
            90deg,
            #365900,
            #b9ae00,
            #fff200,
            #53b932,
            #315800
        );

    background-size: 300% auto;

    -webkit-background-clip: text;

    -webkit-text-fill-color: transparent;

    animation:
        titleGlow 6s ease-in-out infinite;
}


@keyframes titleGlow {

    0% {
        background-position: 0% center;
    }

    50% {
        background-position: 100% center;
    }

    100% {
        background-position: 0% center;
    }
}


/* ============================================================
   HERO DESCRIPTION
   ============================================================ */

.hero p {

    max-width: 720px;

    margin:
        0
        auto;

    color:
        #4f652c !important;

    font-size: 17px;

    line-height: 1.8;

    font-weight: 500;
}


/* ============================================================
   MAIN APPLICATION CARD
   ============================================================ */

.main-card {

    position: relative;

    margin:
        10px
        18px
        35px;

    padding:
        30px;

    border-radius:
        32px;

    background:

        linear-gradient(
            145deg,
            rgba(255,255,235,0.97),
            rgba(255,249,163,0.88)
        ) !important;

    border:
        2px solid
        rgba(214, 211, 60, 0.85) !important;

    box-shadow:

        0 25px 70px
        rgba(118, 145, 40, 0.18),

        inset 0 1px 0
        rgba(255,255,255,0.9);

    backdrop-filter:
        blur(14px);
}


.main-card::before {

    content: "";

    position: absolute;

    top: 0;

    left: 8%;

    width: 84%;

    height: 4px;

    border-radius:
        0 0 10px 10px;

    background:

        linear-gradient(
            90deg,
            #fff200,
            #ffff70,
            #6bd43f,
            #fff200
        );

    box-shadow:
        0 4px 18px
        rgba(200,210,30,0.30);
}


/* ============================================================
   CHATBOT
   ============================================================ */

#chatbot {

    position: relative;

    background:
        #f5fbd3 !important;

    border:
        2px solid
        #8fc95b !important;

    border-radius:
        25px !important;

    overflow:
        hidden !important;

    color:
        #203900 !important;

    box-shadow:

        0 15px 35px
        rgba(90,130,40,0.13),

        inset 0 1px 0
        rgba(255,255,255,0.9) !important;

    --body-text-color:
        #203900 !important;

    --color-text-primary:
        #203900 !important;

    --color-text-body:
        #203900 !important;

    --block-label-text-color:
        #203900 !important;
}


/* ============================================================
   CHATBOT INNER
   ============================================================ */

#chatbot .wrap {

    background:

        linear-gradient(
            135deg,
            #fffdd1 0%,
            #f7fac5 48%,
            #e5f7c8 100%
        ) !important;
}


/* ============================================================
   CHAT TEXT
   ============================================================ */

#chatbot .message,
#chatbot .message-content,
#chatbot .prose {

    color:
        #203900 !important;
}


#chatbot .message *,
#chatbot .message-content *,
#chatbot .prose *,
#chatbot p,
#chatbot span,
#chatbot strong,
#chatbot em,
#chatbot li {

    opacity:
        1 !important;

    visibility:
        visible !important;

    text-shadow:
        none !important;
}


/* ============================================================
   USER MESSAGE
   ============================================================ */

#chatbot .message.user {

    background:

        linear-gradient(
            135deg,
            #fff200 0%,
            #fff94d 55%,
            #eaff70 100%
        ) !important;

    border:
        2px solid
        #c8ca00 !important;

    border-radius:
        20px 20px 6px 20px !important;

    color:
        #243400 !important;

    font-size:
        17px !important;

    font-weight:
        750 !important;

    padding:
        15px
        19px !important;

    box-shadow:

        0 7px 18px
        rgba(190,185,0,0.20) !important;
}


#chatbot .message.user *,
#chatbot .message.user .message-content *,
#chatbot .message.user .prose * {

    color:
        #243400 !important;

    font-size:
        17px !important;

    font-weight:
        750 !important;
}


/* ============================================================
   ASSISTANT MESSAGE
   ============================================================ */

#chatbot .message.bot,
#chatbot .message.assistant {

    background:

        linear-gradient(
            135deg,
            #d7f78c 0%,
            #b8ee72 50%,
            #91dc5b 100%
        ) !important;

    border:
        2px solid
        #70bc46 !important;

    border-radius:
        20px 20px 20px 6px !important;

    color:
        #173700 !important;

    font-size:
        17px !important;

    font-weight:
        600 !important;

    padding:
        15px
        19px !important;

    box-shadow:

        0 7px 18px
        rgba(75,150,45,0.18) !important;
}


#chatbot .message.bot *,
#chatbot .message.assistant *,
#chatbot .message.bot .message-content *,
#chatbot .message.assistant .message-content *,
#chatbot .message.bot .prose *,
#chatbot .message.assistant .prose * {

    color:
        #173700 !important;

    font-size:
        17px !important;

    font-weight:
        600 !important;
}


/* ============================================================
   CHAT MARKDOWN
   ============================================================ */

#chatbot .prose p {

    color:
        #173700 !important;

    font-size:
        17px !important;

    line-height:
        1.7 !important;

    margin-bottom:
        8px !important;
}


#chatbot .prose strong {

    color:
        #477d14 !important;

    font-weight:
        900 !important;
}


#chatbot .prose code {

    display:
        inline-block;

    color:
        #243400 !important;

    background:
        #fff200 !important;

    padding:
        4px 8px !important;

    border-radius:
        7px !important;

    font-weight:
        900 !important;

    border:
        1px solid
        #d4d000 !important;
}


/* ============================================================
   PLACEHOLDER
   ============================================================ */

#chatbot .placeholder {

    color:
        #60743d !important;

    font-size:
        16px !important;

    opacity:
        1 !important;
}


/* ============================================================
   TEXT INPUT
   ============================================================ */

textarea {

    background:

        linear-gradient(
            135deg,
            #fbffd9,
            #ecf9bd
        ) !important;

    color:
        #203900 !important;

    border:
        2px solid
        #91ca5a !important;

    border-radius:
        18px !important;

    font-size:
        16px !important;

    font-weight:
        600 !important;

    padding:
        15px
        17px !important;

    box-shadow:
        inset 0 2px 8px
        rgba(100,140,50,0.06) !important;

    transition:
        all 0.25s ease !important;
}


textarea:hover {

    border-color:
        #76bd45 !important;
}


textarea:focus {

    border-color:
        #64ad3b !important;

    box-shadow:

        0 0 0 4px
        rgba(103,190,60,0.18),

        0 8px 20px
        rgba(80,140,40,0.10) !important;
}


textarea::placeholder {

    color:
        #64783f !important;

    opacity:
        1 !important;
}


/* ============================================================
   SEND BUTTON
   ============================================================ */

.send-button {

    min-height:
        58px !important;

    border:
        2px solid
        #c6c600 !important;

    border-radius:
        18px !important;

    background:

        linear-gradient(
            135deg,
            #fff200 0%,
            #ffff52 50%,
            #e6ff56 100%
        ) !important;

    color:
        #263b00 !important;

    font-size:
        16px !important;

    font-weight:
        900 !important;

    box-shadow:

        0 10px 25px
        rgba(180,180,0,0.22) !important;

    transition:
        all 0.25s ease !important;
}


.send-button:hover {

    transform:
        translateY(-4px) !important;

    background:

        linear-gradient(
            135deg,
            #ffff00,
            #eaff45
        ) !important;

    box-shadow:

        0 15px 30px
        rgba(170,180,0,0.28) !important;
}


.send-button:active {

    transform:
        translateY(0)
        scale(0.98) !important;
}


/* ============================================================
   CLEAR BUTTON
   ============================================================ */

.clear-button {

    min-height:
        48px !important;

    margin-top:
        10px !important;

    border-radius:
        16px !important;

    background:

        linear-gradient(
            135deg,
            #d9f99d,
            #a9e96b
        ) !important;

    border:
        2px solid
        #78bd47 !important;

    color:
        #244100 !important;

    font-weight:
        800 !important;

    transition:
        all 0.25s ease !important;
}


.clear-button:hover {

    transform:
        translateY(-2px) !important;

    background:
        #baf080 !important;

    box-shadow:

        0 8px 20px
        rgba(70,140,40,0.18) !important;
}


/* ============================================================
   MATCH INFORMATION
   ============================================================ */

.info-box {

    margin-top:
        20px !important;

    padding:
        20px 22px !important;

    border-radius:
        20px !important;

    background:

        linear-gradient(
            135deg,
            #fff79c,
            #e9f8b8,
            #c9f185
        ) !important;

    border:
        2px solid
        #b8cd51 !important;

    color:
        #304800 !important;

    box-shadow:

        0 10px 25px
        rgba(130,150,40,0.12) !important;
}


.info-box,
.info-box * {

    color:
        #304800 !important;
}


.info-box h3 {

    color:
        #466700 !important;

    margin-top:
        0 !important;
}


.info-box strong {

    color:
        #507600 !important;

    font-weight:
        900 !important;
}


.info-box code {

    background:
        #fff000 !important;

    color:
        #263700 !important;

    border:
        1px solid
        #d3ce00 !important;

    border-radius:
        7px !important;

    padding:
        4px 8px !important;

    font-weight:
        900 !important;
}


/* ============================================================
   EXAMPLE TITLE
   ============================================================ */

.main-card h3 {

    color:
        #507000 !important;

    font-size:
        19px !important;

    font-weight:
        900 !important;

    margin-top:
        24px !important;
}


/* ============================================================
   EXAMPLE BUTTONS
   ============================================================ */

.main-card button {

    color:
        #294300 !important;

    background:

        linear-gradient(
            135deg,
            #e5f8a5,
            #c9ef82
        ) !important;

    border:
        2px solid
        #93c85b !important;

    border-radius:
        14px !important;

    font-weight:
        700 !important;

    transition:
        all 0.22s ease !important;
}


.main-card button:hover {

    transform:
        translateY(-2px) !important;

    color:
        #203600 !important;

    background:
        #fff000 !important;

    border-color:
        #c5c500 !important;

    box-shadow:

        0 7px 18px
        rgba(180,180,0,0.17) !important;
}


/* ============================================================
   FEATURE SECTION
   ============================================================ */

.features {

    display:
        grid;

    grid-template-columns:
        repeat(3, 1fr);

    gap:
        18px;

    margin-top:
        30px;
}


/* ============================================================
   FEATURE CARD
   ============================================================ */

.feature {

    position:
        relative;

    padding:
        25px 18px;

    text-align:
        center;

    border-radius:
        22px;

    background:

        linear-gradient(
            145deg,
            #fff7a0,
            #eaf8b9,
            #c5f184
        );

    border:
        2px solid
        #afd05c;

    box-shadow:

        0 10px 25px
        rgba(120,145,45,0.10);

    overflow:
        hidden;

    transition:
        transform 0.3s ease,
        box-shadow 0.3s ease,
        border-color 0.3s ease;
}


.feature::before {

    content:
        "";

    position:
        absolute;

    width:
        100px;

    height:
        100px;

    border-radius:
        50%;

    background:
        rgba(255,255,255,0.30);

    top:
        -55px;

    right:
        -35px;
}


.feature:hover {

    transform:
        translateY(-8px);

    border-color:
        #72b943;

    box-shadow:

        0 18px 35px
        rgba(80,140,40,0.18);
}


/* ============================================================
   FEATURE ICON
   ============================================================ */

.feature-icon {

    display:
        flex;

    align-items:
        center;

    justify-content:
        center;

    width:
        58px;

    height:
        58px;

    margin:
        0 auto 13px;

    border-radius:
        18px;

    background:
        #fff200;

    border:
        2px solid
        #d0cc00;

    font-size:
        29px;

    box-shadow:

        0 7px 15px
        rgba(170,170,0,0.18);
}


/* ============================================================
   FEATURE TITLE
   ============================================================ */

.feature-title {

    color:
        #294400 !important;

    font-size:
        17px;

    font-weight:
        900;

    margin-top:
        5px;
}


/* ============================================================
   FEATURE TEXT
   ============================================================ */

.feature-text {

    color:
        #5a703b !important;

    font-size:
        13px;

    line-height:
        1.5;

    margin-top:
        7px;
}


/* ============================================================
   FOOTER
   ============================================================ */

.footer {

    text-align:
        center;

    padding:
        25px
        20px
        45px;

    color:
        #697743 !important;

    font-size:
        13px;

    line-height:
        1.7;
}


.footer strong {

    color:
        #4e7100 !important;

    font-weight:
        900 !important;
}


/* ============================================================
   SCROLLBAR
   ============================================================ */

::-webkit-scrollbar {

    width:
        8px;
}


::-webkit-scrollbar-track {

    background:
        #f4f7c7;

    border-radius:
        10px;
}


::-webkit-scrollbar-thumb {

    background:

        linear-gradient(
            180deg,
            #dede00,
            #7bc84a
        );

    border-radius:
        10px;
}


::-webkit-scrollbar-thumb:hover {

    background:
        #6cbb3d;
}


/* ============================================================
   TEXT SELECTION
   ============================================================ */

::selection {

    background:
        #fff200;

    color:
        #243900;
}


/* ============================================================
   TABLET
   ============================================================ */

@media (max-width: 850px) {

    .hero {

        padding:
            45px
            15px
            30px;
    }

    .hero h1 {

        font-size:
            43px;
    }

    .main-card {

        margin:
            8px
            10px
            25px;

        padding:
            18px;

        border-radius:
            25px;
    }

    .features {

        grid-template-columns:
            1fr;
    }
}


/* ============================================================
   MOBILE
   ============================================================ */

@media (max-width: 600px) {

    .hero h1 {

        font-size:
            35px;

        letter-spacing:
            -0.5px;
    }

    .hero p {

        font-size:
            14px;
    }

    .badge {

        font-size:
            10px;

        padding:
            8px
            14px;
    }

    #chatbot .message,
    #chatbot .message * {

        font-size:
            15px !important;
    }

    .send-button {

        min-height:
            52px !important;
    }

    .feature {

        padding:
            22px
            15px;
    }
}

"""


# ============================================================
# GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    title="AI FAQ Assistant"
) as app:


    # ========================================================
    # HERO
    # ========================================================

    gr.HTML("""

        <div class="hero">

            <div class="badge">
                ✦ CODEALPHA AI INTERNSHIP PROJECT
            </div>

            <h1>
                🤖 AI FAQ Assistant
            </h1>

            <p>
                Smart answers. Simple experience.
                Ask questions naturally and get instant answers
                using NLP and intelligent FAQ matching.
            </p>

        </div>

    """)


    # ========================================================
    # MAIN CARD
    # ========================================================

    with gr.Column(
        elem_classes="main-card"
    ):


        # ====================================================
        # CHATBOT
        # ====================================================

        chatbot = gr.Chatbot(

            label="💬 Conversation",


            height=430,

            elem_id="chatbot",

            elem_classes="chat-area",

            render_markdown=True,

            placeholder=
                "💬 Ask a question to start chatting..."

        )


        # ====================================================
        # INPUT
        # ====================================================

        with gr.Row():

            message = gr.Textbox(

                placeholder=
                    "Ask me something like: What are the library timings?",

                show_label=False,

                scale=5,

                lines=2

            )

            send_button = gr.Button(

                "🚀 Send",

                scale=1,

                elem_classes="send-button"

            )


        # ====================================================
        # CLEAR
        # ====================================================

        clear_button = gr.Button(

            "🗑️ Clear Conversation",

            elem_classes="clear-button"

        )


        # ====================================================
        # MATCH INFORMATION
        # ====================================================

        match_info = gr.Markdown(

            "💬 Ask a question to get started.",

            elem_classes="info-box"

        )


        # ====================================================
        # EXAMPLES
        # ====================================================

        gr.Markdown(
            "### 💡 Try an Example"
        )

        gr.Examples(

            examples=[

                ["What are the college timings?"],

                ["When does the library open?"],

                ["How can I check my attendance?"],

                ["How do I get my student ID card?"],

                ["Where can I find the exam schedule?"],

                ["How can I pay my college fees?"]

            ],

            inputs=message

        )


        # ====================================================
        # FEATURES
        # ====================================================

        gr.HTML("""

            <div class="features">

                <div class="feature">

                    <div class="feature-icon">
                        🧠
                    </div>

                    <div class="feature-title">
                        NLP Powered
                    </div>

                    <div class="feature-text">
                        Text preprocessing and analysis
                    </div>

                </div>


                <div class="feature">

                    <div class="feature-icon">
                        🎯
                    </div>

                    <div class="feature-title">
                        Smart Matching
                    </div>

                    <div class="feature-text">
                        TF-IDF + cosine similarity
                    </div>

                </div>


                <div class="feature">

                    <div class="feature-icon">
                        ⚡
                    </div>

                    <div class="feature-title">
                        Instant Answers
                    </div>

                    <div class="feature-text">
                        Quick FAQ responses
                    </div>

                </div>

            </div>

        """)


    # ========================================================
    # FOOTER
    # ========================================================

    gr.HTML("""

        <div class="footer">

            Built with
            Python • NLTK • Scikit-learn • Gradio

            <br><br>

            <strong>
                CodeAlpha Artificial Intelligence Internship
            </strong>

        </div>

    """)


    # ========================================================
    # EVENTS
    # ========================================================

    send_button.click(

        fn=respond,

        inputs=[
            message,
            chatbot
        ],

        outputs=[
            chatbot,
            message,
            match_info
        ]

    )


    message.submit(

        fn=respond,

        inputs=[
            message,
            chatbot
        ],

        outputs=[
            chatbot,
            message,
            match_info
        ]

    )


    clear_button.click(

        fn=clear_chat,

        inputs=[],

        outputs=[
            chatbot,
            message,
            match_info
        ]

    )


# ============================================================
# LAUNCH
# ============================================================

app.launch(
    css=custom_css,
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dddbc3b39b5c20b945.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
